# Kubernetes na GCP (GKE): samonaprawa, skalowanie i aktualizacje bez przestojów

Demo na klastrze **Google Kubernetes Engine (Standard)** z 3 węzłami i włączonym **Cluster Autoscalerem**.

**Plan pokazu**

| # | Temat | Co zobaczymy |
|---|-------|--------------|
| 1 | Klaster GKE | `gcloud container clusters create`, węzły, metryki |
| 2 | Aplikacja Fibonacci: Deployment + Service (LoadBalancer) | stan deklaratywny, rozkład podów po węzłach, publiczny IP i load balancing |
| 3 | **Samonaprawa** | usuwamy poda, „wyłączamy” cały węzeł (`drain`) |
| 4 | **Skalowanie ręczne** | `kubectl scale` 3 → 12 → 3 i czas reakcji |
| 5 | **Autoskalowanie podów (HPA)** | zmieniamy ruch (req/s) i ciężar zapytań (`n`) → pody dochodzą same, wykres na żywo |
| 6 | **Autoskalowanie węzłów (Cluster Autoscaler)** | brak miejsca → GKE sam dokłada maszyny wirtualne |
| 7 | **Rolling update i rollback** | nowa wersja bez przestoju, zepsuta wersja, `rollout undo` |
| 8 | Sprzątanie | usunięcie klastra (ważne, bo to kosztuje!) |

**Najważniejsza idea:** w Kubernetesie opisujemy **stan pożądany** (np. „3 repliki tej aplikacji”), a kontrolery w pętli
porównują go ze stanem faktycznym i naprawiają różnice. Z tej jednej idei wynika wszystko, co pokazujemy niżej.

```
Deployment ──► ReplicaSet ──► Pod, Pod, Pod      ◄── Service (stały adres + load balancing)
     ▲                         │
     │                         └── działają na węzłach = maszynach wirtualnych Compute Engine
     └── HorizontalPodAutoscaler                       ▲
         (liczba podów wg metryk)                       └── Cluster Autoscaler (liczba węzłów)
```

**Wymagania:** `gcloud` (zalogowany: `gcloud auth login`, ustawiony projekt), `kubectl`, `gke-gcloud-auth-plugin`, Python z `matplotlib`.
Na macOS: `brew install --cask google-cloud-sdk`, potem `gcloud components install kubectl gke-gcloud-auth-plugin`.

> 💰 **Koszt:** 3–5 × `e2-standard-2` (≈ 0,07 USD/h za węzeł) + load balancer. Opłatę za zarządzanie jednym klastrem
> strefowym pokrywa free tier GKE. Cały pokaz to ok. 1 USD, **o ile na końcu usuniesz klaster** (sekcja 8).

### Konfiguracja
Zmienne z tej komórki trafiają do środowiska, więc widzą je wszystkie komórki `%%bash`.
Komórka dopisuje też do `PATH` katalog `bin` z Google Cloud SDK, bo tam są `kubectl` i `gke-gcloud-auth-plugin`.
**Uruchom ją jako pierwszą** (także po restarcie kernela).

> ⚠️ **Free trial / konta edukacyjne** mają limit 8 vCPU (quota `CPUS`). Wtedy ustaw `MAX_NODES = 4`.

In [ ]:
import os, subprocess, shutil, sys

CLUSTER   = "sgh-demo"
ZONE      = "europe-central2-a"      # Warszawa
MACHINE   = "e2-standard-2"          # 2 vCPU, 8 GB RAM
NODES     = 3                        # węzły na start
MAX_NODES = 5                        # górny limit dla Cluster Autoscalera

def prepend_path(d):
    if d and os.path.isdir(d) and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]

# Windows: komórki %%bash mają używać Git Bash, a nie WSL (WindowsApps\bash.exe uruchamia Ubuntu,
# które nie widzi gcloud/kubectl z Windows ani zmiennych środowiskowych z tej komórki).
if os.name == "nt":
    for git_bin in (r"C:\Program Files\Git\bin", r"C:\Program Files (x86)\Git\bin",
                    os.path.expandvars(r"%LOCALAPPDATA%\Programs\Git\bin")):
        if os.path.exists(os.path.join(git_bin, "bash.exe")):
            prepend_path(git_bin)
            break

# Na Windows gcloud to plik gcloud.cmd, którego subprocess nie znajdzie po samej nazwie "gcloud",
# dlatego pełną ścieżkę wyszukujemy przez shutil.which (uwzględnia PATHEXT).
GCLOUD = shutil.which("gcloud")
if not GCLOUD:
    raise FileNotFoundError("Nie znaleziono gcloud w PATH. Zainstaluj Google Cloud SDK i zrestartuj VS Code/kernel.")

# kubectl i gke-gcloud-auth-plugin instalują się w <SDK>/bin, ale Homebrew linkuje do PATH tylko gcloud,
# więc dopisujemy ten katalog do PATH (działa i dla komórek %%bash, i dla subprocess w Pythonie).
SDK_ROOT = subprocess.run([GCLOUD, "info", "--format=value(installation.sdk_root)"], capture_output=True, text=True).stdout.strip()
prepend_path(os.path.join(SDK_ROOT, "bin") if SDK_ROOT else None)
print("gcloud: ", GCLOUD)
print("kubectl:", shutil.which("kubectl") or "❌ brak, uruchom: gcloud components install kubectl gke-gcloud-auth-plugin")
print("bash:   ", shutil.which("bash"))

PROJECT = subprocess.run([GCLOUD, "config", "get-value", "project"], capture_output=True, text=True).stdout.strip()
os.environ.update(CLUSTER=CLUSTER, ZONE=ZONE, MACHINE=MACHINE, NODES=str(NODES), MAX_NODES=str(MAX_NODES), PROJECT=PROJECT,
                  PYTHON=sys.executable.replace("\\", "/"),   # interpreter kernela, dla komórek %%bash
                  PYTHONUTF8="1")                            # emoji w print() z procesów potomnych (Windows: cp1252)
os.makedirs("k8s-demo", exist_ok=True)
print(f"Projekt: {PROJECT or '❌ BRAK, ustaw: gcloud config set project <ID>'}   strefa: {ZONE}   węzły: {NODES}..{MAX_NODES} × {MACHINE}")

In [ ]:
%%bash
for t in gcloud kubectl gke-gcloud-auth-plugin; do
  command -v $t >/dev/null && echo "✅ $t  ($(command -v $t))" || echo "❌ brak: $t   (gcloud components install $t)"
done
ACCOUNT=$(gcloud config get-value account 2>/dev/null)
[ -n "$ACCOUNT" ] && echo "✅ zalogowany jako: $ACCOUNT" || echo "❌ gcloud auth login"
# $PYTHON = interpreter kernela (ustawiony w komórce konfiguracyjnej); samo "python3" na Windows to często alias Microsoft Store
"$PYTHON" -c "import matplotlib; print('✅ matplotlib', matplotlib.__version__)" || echo "❌ \"$PYTHON\" -m pip install matplotlib"

---
## 1. Klaster GKE

Włączamy API Kubernetes Engine i tworzymy klaster strefowy (tworzenie trwa **ok. 5–8 minut**, warto to zrobić przed pokazem):
- `--enable-autoscaling --min-nodes/--max-nodes` włącza **Cluster Autoscaler** dla puli węzłów,
- `--disk-type pd-standard --disk-size 30` to mniejsze dyski, żeby nie przekroczyć limitu SSD w projekcie.

GKE ma wbudowany **metrics-server** (zużycie CPU i RAM podów), więc HPA działa od razu.

In [ ]:
%%bash
set -e
gcloud services enable container.googleapis.com --project "$PROJECT"
if gcloud container clusters describe "$CLUSTER" --zone "$ZONE" --project "$PROJECT" >/dev/null 2>&1; then
  echo "Klaster $CLUSTER już istnieje"
else
  gcloud container clusters create "$CLUSTER" \
    --project "$PROJECT" --zone "$ZONE" \
    --machine-type "$MACHINE" --num-nodes "$NODES" \
    --enable-autoscaling --min-nodes "$NODES" --max-nodes "$MAX_NODES" \
    --disk-type pd-standard --disk-size 30 \
    --release-channel regular
fi
gcloud container clusters get-credentials "$CLUSTER" --zone "$ZONE" --project "$PROJECT"


Każdy węzeł to zwykła maszyna wirtualna Compute Engine, zarządzana przez GKE:

In [ ]:
%%bash
gcloud compute instances list --project "$PROJECT" --filter="name~^gke-$CLUSTER" \
  --format="table(name, zone.basename(), machineType.basename(), status)"
echo
echo "Czekam na metryki..."
for i in $(seq 1 30); do kubectl top nodes >/dev/null 2>&1 && break; sleep 5; done
kubectl top nodes

### Funkcje pomocnicze do wizualizacji

`watch_pods()` rysuje w terminalu, na którym węźle działa który pod:
🟢 Ready · 🟡 uruchamia się · ⚪ Pending (brak miejsca) · 🔴 Terminating · 🟠 błąd pobierania obrazu

In [3]:
import json, subprocess, time
from collections import Counter
from IPython.display import clear_output
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

NS = "k8s-demo"

def kubectl(*args, ns=True, check=True, input=None):
    cmd = ["kubectl"] + (["-n", NS] if ns else []) + [str(a) for a in args]
    r = subprocess.run(cmd, capture_output=True, text=True, input=input)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    return r.stdout

def kjson(*args, ns=True):
    return json.loads(kubectl(*args, "-o", "json", ns=ns))

def is_control_plane(node):
    return "node-role.kubernetes.io/control-plane" in node["metadata"].get("labels", {})

def worker_nodes():
    return [n for n in kjson("get", "nodes", ns=False)["items"] if not is_control_plane(n)]

def pod_state(p):
    if p["metadata"].get("deletionTimestamp"):
        return "Terminating"
    st = p.get("status", {})
    css = st.get("containerStatuses") or []
    for cs in css:
        waiting = (cs.get("state") or {}).get("waiting")
        if waiting:
            return waiting.get("reason", "Waiting")
    if st.get("phase") == "Running":
        return "Ready" if css and all(cs.get("ready") for cs in css) else "Starting"
    return st.get("phase", "?")

ICONS = {"Ready": "🟢", "Starting": "🟡", "ContainerCreating": "🟡", "Pending": "⚪",
         "Terminating": "🔴", "ErrImagePull": "🟠", "ImagePullBackOff": "🟠"}
def icon(state):
    return ICONS.get(state, "❔")

def pods_by_node(selector):
    nodes = kjson("get", "nodes", ns=False)["items"]
    by_node, labels = {}, {}
    for n in nodes:
        name = n["metadata"]["name"]
        by_node[name] = []
        cordoned = " [cordoned]" if n["spec"].get("unschedulable") else ""
        ready = any(c["type"] == "Ready" and c["status"] == "True" for c in n["status"].get("conditions", []))
        labels[name] = name + (" (control-plane)" if is_control_plane(n) else "") + cordoned + ("" if ready else " [NotReady]")
    pods = kjson("get", "pods", "-l", selector)["items"]
    for p in pods:
        node = p["spec"].get("nodeName") or "⏳ nieprzydzielone"
        labels.setdefault(node, node)
        by_node.setdefault(node, []).append(p)
    return by_node, labels, pods

def render_pods(selector, show_image=False):
    by_node, labels, pods = pods_by_node(selector)
    width = max(len(l) for l in labels.values())
    lines = []
    for node, ps in by_node.items():
        ps.sort(key=lambda p: p["metadata"]["creationTimestamp"])
        if show_image:
            cells = " ".join(icon(pod_state(p)) + p["spec"]["containers"][0]["image"].split(":")[-1].replace("-alpine", "")
                             for p in ps)
        else:
            cells = "".join(icon(pod_state(p)) for p in ps)
        lines.append(f"{labels[node]:<{width}} │ {cells}")
    counts = Counter(pod_state(p) for p in pods)
    summary = "   ".join(f"{icon(s)} {s}: {c}" for s, c in sorted(counts.items()))
    lines.append(f"\nPodów: {len(pods)}   {summary}")
    return "\n".join(lines), pods

def watch_pods(selector, seconds=30, interval=1.0, title="", show_image=False, target=None):
    """Odświeża widok podów. Gdy podano target, kończy, kiedy jest dokładnie `target` podów i wszystkie są Ready."""
    t0 = time.time()
    while True:
        out, pods = render_pods(selector, show_image)
        elapsed = time.time() - t0
        clear_output(wait=True)
        print(f"{title}   ⏱ {elapsed:5.1f}s\n")
        print(out)
        if target is not None and len(pods) == target and all(pod_state(p) == "Ready" for p in pods):
            print(f"\n✅ Stan pożądany osiągnięty po {elapsed:.1f}s")
            return
        if elapsed >= seconds:
            return
        time.sleep(interval)

print("OK, kontekst:", kubectl("config", "current-context", ns=False).strip())

OK, kontekst: gke_braided-triode-508616-i5_europe-central2-a_sgh-demo


---
## 2. Aplikacja: Fibonacci jako usługa

Przykładowa aplikacja liczy **n-ty wyraz ciągu Fibonacciego** metodą *celowo naiwną* (rekurencja bez pamięci):

```python
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
```

Liczba wywołań rośnie wykładniczo (~1,6ⁿ), więc **parametrem `n` sterujemy kosztem CPU jednego żądania**:
`fib(25)` to kilkadziesiąt ms, `fib(30)` to już sekundy. Dzięki temu łatwo pokazać zmianę obciążenia:
- **więcej żądań** (req/s) → więcej pracy,
- **cięższe żądania** (większe `n`) → dużo więcej pracy przy tej samej liczbie żądań.

Endpointy:
| URL | Co robi |
|---|---|
| `/` | strona WWW: formularz + tryb „ruch ciągły”, który pokazuje, który pod odpowiada |
| `/fib?n=25` | JSON: `{"n": 25, "fib": 75025, "ms": 41.3, "pod": "fib-app-…"}` |
| `/healthz` | sonda gotowości (readinessProbe) |

Kod trafia do kontenera przez **ConfigMap**, więc nie trzeba budować obrazu (wystarczy publiczny `python:3.13-alpine`).

Ważne fragmenty manifestu:
- `replicas: 3`, czyli stan pożądany,
- `resources.requests` / `limits`: ile CPU pod rezerwuje i ile maksymalnie może zużyć (od tego liczy się % w HPA),
- `readinessProbe`: ruch trafia tylko do podów, które odpowiadają na `/healthz`,
- Service `type: LoadBalancer`: GKE sam tworzy **Google Cloud Load Balancer** z publicznym IP.

In [ ]:
%%writefile k8s-demo/app.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: fib-app-code
data:
  app.py: |
    import http.server, json, socket, time
    from urllib.parse import urlparse, parse_qs

    POD = socket.gethostname()
    MAX_N = 35

    def fib(n):
        # Celowo naiwnie: O(1.6^n) wywołań. Tym obciążamy CPU.
        return n if n < 2 else fib(n - 1) + fib(n - 2)

    PAGE = """<!doctype html>
    <html lang="pl"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>Fibonacci na Kubernetesie</title>
    <style>
     body{font-family:system-ui,sans-serif;max-width:760px;margin:2rem auto;padding:0 1rem;background:#0f172a;color:#e2e8f0}
     h1{font-size:1.6rem} input,button{font-size:1.1rem;padding:.4rem .8rem;border-radius:6px;border:1px solid #475569}
     input{width:5rem;background:#1e293b;color:#e2e8f0} button{background:#2563eb;color:#fff;cursor:pointer}
     #out{font-size:1.3rem;margin:1rem 0} .pod{font-family:monospace;color:#facc15}
     table{border-collapse:collapse;width:100%} td{padding:.2rem .5rem;border-bottom:1px solid #334155;font-family:monospace}
     .bar{background:#22c55e;height:.8rem;border-radius:3px}
    </style></head><body>
    <h1>🧮 Fibonacci na Kubernetesie</h1>
    <p>Stronę wysłał pod: <span class="pod">{POD}</span></p>
    <p>n = <input id="n" type="number" value="25" min="0" max="35">
       <button onclick="once()">Licz</button>
       <button id="loop" onclick="toggle()">▶ Ruch ciągły</button></p>
    <div id="out"></div>
    <table id="pods"></table>
    <script>
    const tally = {}; let timer = null;
    async function call() {
      const n = document.getElementById('n').value;
      const r = await fetch('/fib?n=' + n, {cache: 'no-store'});
      const d = await r.json();
      if (d.error) { document.getElementById('out').textContent = d.error; return; }
      document.getElementById('out').innerHTML =
        `fib(${d.n}) = <b>${d.fib}</b> &nbsp; ${d.ms} ms &nbsp; pod: <span class="pod">${d.pod}</span>`;
      tally[d.pod] = (tally[d.pod] || 0) + 1;
      const max = Math.max(...Object.values(tally));
      document.getElementById('pods').innerHTML = Object.entries(tally).sort()
        .map(([p, c]) => `<tr><td class="pod">${p}</td><td>${c}</td><td style="width:50%"><div class="bar" style="width:${100*c/max}%"></div></td></tr>`).join('');
    }
    function once() { call(); }
    function toggle() {
      const b = document.getElementById('loop');
      if (timer) { clearInterval(timer); timer = null; b.textContent = '▶ Ruch ciągły'; }
      else { timer = setInterval(call, 300); b.textContent = '⏸ Stop'; }
    }
    </script></body></html>"""

    class Handler(http.server.BaseHTTPRequestHandler):
        # HTTP/1.0: każde żądanie to nowe połączenie, więc load balancer rozkłada ruch na różne pody
        def send(self, code, body, ctype):
            self.send_response(code)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            url = urlparse(self.path)
            if url.path == "/healthz":
                return self.send(200, b"ok", "text/plain")
            if url.path == "/fib":
                try:
                    n = int(parse_qs(url.query).get("n", ["25"])[0])
                    assert 0 <= n <= MAX_N
                except (ValueError, AssertionError):
                    err = json.dumps({"error": f"n musi być liczbą 0..{MAX_N}"}).encode()
                    return self.send(400, err, "application/json")
                t = time.perf_counter()
                result = fib(n)
                ms = round((time.perf_counter() - t) * 1000, 1)
                body = json.dumps({"n": n, "fib": result, "ms": ms, "pod": POD}).encode()
                return self.send(200, body, "application/json")
            if url.path == "/":
                return self.send(200, PAGE.replace("{POD}", POD).encode(), "text/html; charset=utf-8")
            self.send(404, b"not found", "text/plain")

        def log_message(self, *args):
            pass

    print(f"fib-app startuje na porcie 8080, pod {POD}")
    http.server.ThreadingHTTPServer(("", 8080), Handler).serve_forever()
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fib-app
spec:
  replicas: 3
  selector:
    matchLabels: {app: fib-app}
  template:
    metadata:
      labels: {app: fib-app}
    spec:
      containers:
        - name: app
          image: python:3.13-alpine
          command: ["python", "-u", "/app/app.py"]
          ports: [{containerPort: 8080}]
          readinessProbe:
            httpGet: {path: /healthz, port: 8080}
            periodSeconds: 2
          resources:
            requests: {cpu: 200m, memory: 32Mi}
            limits:   {cpu: 500m, memory: 128Mi}
          volumeMounts: [{name: code, mountPath: /app}]
      volumes:
        - name: code
          configMap: {name: fib-app-code}
---
apiVersion: v1
kind: Service
metadata:
  name: fib-app
spec:
  type: LoadBalancer
  selector: {app: fib-app}
  ports: [{port: 80, targetPort: 8080}]

In [ ]:
%%bash
kubectl create namespace k8s-demo --dry-run=client -o yaml | kubectl apply -f -
kubectl -n k8s-demo apply -f k8s-demo/app.yaml
kubectl -n k8s-demo rollout status deployment/fib-app --timeout=180s
echo
kubectl -n k8s-demo get deployment,replicaset,pods,service -o wide

Scheduler sam rozłożył pody po węzłach. Control-plane w GKE jest zarządzany przez Google i nie widać go na liście węzłów:

In [ ]:
print(render_pods("app=fib-app")[0])

**Service = stały adres + load balancing.** Pody przychodzą i odchodzą (zmieniają się ich IP), a adres usługi zostaje ten sam.
Czekamy na publiczny IP load balancera (zwykle 1–2 minuty).

👉 **Otwórz wypisany adres w przeglądarce** (np. na rzutniku) i włącz „▶ Ruch ciągły”. Na żywo widać, które pody odpowiadają.
Najlepiej zostawić tę kartę otwartą na cały pokaz: przy skalowaniu i samonaprawie pojawiają się i znikają nazwy podów.

In [ ]:
%%bash
echo -n "Czekam na zewnętrzny IP"
for i in $(seq 1 60); do
  IP=$(kubectl -n k8s-demo get svc fib-app -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
  [ -n "$IP" ] && break; echo -n "."; sleep 5
done
echo; echo "🌍 http://$IP"
for i in $(seq 1 60); do curl -s -m 2 -o /dev/null "http://$IP/healthz" && break; sleep 3; done
echo; echo "30 żądań z laptopa, kto odpowiada:"
for i in $(seq 1 30); do curl -s "http://$IP/fib?n=10"; echo; done | grep -o '"pod": "[^"]*"' | sort | uniq -c

**Koszt rośnie wykładniczo z `n`.** Każde +1 do `n` to ok. ×1,6 czasu CPU. To główna „gałka” do sterowania obciążeniem:

In [ ]:
%%bash
IP=$(kubectl -n k8s-demo get svc fib-app -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
for n in 10 20 24 26 28 30; do curl -s "http://$IP/fib?n=$n"; echo; done

---
## 3. Samonaprawa (self-healing)

### 3a. Usuwamy poda
ReplicaSet widzi „mam 2, a ma być 3” i od razu tworzy nowego poda.

In [ ]:
victim = kjson("get", "pods", "-l", "app=fib-app")["items"][0]["metadata"]["name"]
kubectl("delete", "pod", victim, "--wait=false")
watch_pods("app=fib-app", seconds=40, target=3, title=f"💥 Usunięto pod {victim}, obserwujemy samonaprawę")

### 3b. Awaria / serwis całego węzła
`kubectl drain` wyrzuca wszystkie pody z węzła i oznacza go jako niedostępny (*cordon*), tak jak przy aktualizacji
systemu lub awarii maszyny. Kubernetes odtwarza pody na pozostałych węzłach.

In [ ]:
by_node, _, _ = pods_by_node("app=fib-app")
workers = [n["metadata"]["name"] for n in worker_nodes()]
drained = max(workers, key=lambda n: len(by_node.get(n, [])))

proc = subprocess.Popen(["kubectl", "drain", drained, "--ignore-daemonsets", "--delete-emptydir-data", "--timeout=120s"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
watch_pods("app=fib-app", seconds=30, title=f"🔧 kubectl drain {drained}")
proc.wait()
print("\n" + kubectl("get", "nodes", ns=False))

Przywracamy węzeł (`uncordon`). **Uwaga:** działające pody *nie wracają* same na ten węzeł. Scheduler użyje go
dla **nowych** podów, co zobaczymy przy skalowaniu.

In [ ]:
print(kubectl("uncordon", drained, ns=False))
print(render_pods("app=fib-app")[0])

---
## 4. Skalowanie ręczne

Jedna komenda zmienia stan pożądany. Resztę robi klaster: tworzy pody, rozkłada je po węzłach i dopisuje do Service.

In [ ]:
kubectl("scale", "deployment/fib-app", "--replicas=12")
watch_pods("app=fib-app", seconds=120, target=12, title="📈 kubectl scale deployment/fib-app --replicas=12")

In [ ]:
%%bash
# Load balancing obejmuje teraz wszystkie 12 replik
IP=$(kubectl -n k8s-demo get svc fib-app -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
for i in $(seq 1 60); do curl -s "http://$IP/fib?n=10"; echo; done | grep -o '"pod": "[^"]*"' | sort | uniq -c

In [ ]:
kubectl("scale", "deployment/fib-app", "--replicas=3")
watch_pods("app=fib-app", seconds=60, target=3, title="📉 kubectl scale deployment/fib-app --replicas=3")

---
## 5. Autoskalowanie: HorizontalPodAutoscaler (HPA)

Teraz liczbą replik steruje **HPA**. Co ok. 15 s porównuje średnie zużycie CPU podów z celem (50% `requests`) i liczy:

$$\text{replicas}_{\text{desired}} = \left\lceil \text{replicas}_{\text{current}} \cdot \frac{\text{CPU}_{\text{current}}}{\text{CPU}_{\text{target}}} \right\rceil$$

Sekcja `behavior` przyspiesza demo: skalowanie w górę bez zwłoki, w dół po 30 s spokoju (domyślnie jest to 5 minut).

In [ ]:
%%writefile k8s-demo/hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: fib-app
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: fib-app
  minReplicas: 1
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: {type: Utilization, averageUtilization: 50}
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 0
      policies: [{type: Percent, value: 100, periodSeconds: 15}]
    scaleDown:
      stabilizationWindowSeconds: 30
      policies: [{type: Percent, value: 100, periodSeconds: 15}]
---
# Generator ruchu: wysyła stałą liczbę żądań na sekundę (RPS) o zadanym n.
# Obciążenie ≈ RPS × koszt(fib(n)), więc sterujemy nim dwiema „gałkami”.
apiVersion: v1
kind: ConfigMap
metadata:
  name: load-generator-code
data:
  load.py: |
    import os, time, threading, urllib.request
    from concurrent.futures import ThreadPoolExecutor

    URL = os.environ.get("TARGET", "http://fib-app")
    RPS = float(os.environ.get("RPS", "10"))
    N = int(os.environ.get("FIB_N", "25"))
    MAX_IN_FLIGHT = 64                       # gdy aplikacja nie nadąża, nie zalewamy jej w nieskończoność

    slots = threading.BoundedSemaphore(MAX_IN_FLIGHT)
    pool = ThreadPoolExecutor(max_workers=MAX_IN_FLIGHT)
    stats = {"ok": 0, "err": 0, "skipped": 0}

    def hit():
        try:
            urllib.request.urlopen(f"{URL}/fib?n={N}", timeout=15).read()
            stats["ok"] += 1
        except Exception:
            stats["err"] += 1
        finally:
            slots.release()

    print(f"Generator: {RPS} req/s × fib({N}) → {URL}", flush=True)
    interval, nxt, last = 1.0 / RPS, time.monotonic(), time.monotonic()
    while True:
        if slots.acquire(blocking=False):
            pool.submit(hit)
        else:
            stats["skipped"] += 1
        if time.monotonic() - last >= 10:
            print(stats, flush=True); last = time.monotonic()
        nxt += interval
        time.sleep(max(0.0, nxt - time.monotonic()))
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: load-generator
spec:
  replicas: 0                     # na start ruch wyłączony
  selector:
    matchLabels: {app: load-generator}
  template:
    metadata:
      labels: {app: load-generator}
    spec:
      containers:
        - name: load
          image: python:3.13-alpine
          command: ["python", "-u", "/app/load.py"]
          env:
            - {name: RPS,   value: "5"}
            - {name: FIB_N, value: "25"}
          resources:
            requests: {cpu: 100m, memory: 32Mi}
            limits:   {cpu: 500m, memory: 128Mi}
          volumeMounts: [{name: code, mountPath: /app}]
      volumes:
        - name: code
          configMap: {name: load-generator-code}

In [ ]:
HPA_TARGET = 50
history = []

def _nan(v):
    return float("nan") if v is None else v

def plot_history():
    if not history:
        return
    t0 = history[0]["t"]
    xs = [h["t"] - t0 for h in history]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)

    ax1.step(xs, [_nan(h["current"]) for h in history], where="post", lw=2.5, label="repliki (HPA)")
    ax1.step(xs, [_nan(h["desired"]) for h in history], where="post", ls="--", label="desired (HPA)")
    ax1.plot(xs, [h["ready"] for h in history], "o", ms=4, label="pody Ready")
    ax1.set_ylabel("liczba podów")
    ax1.set_ylim(bottom=0)
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax2.plot(xs, [_nan(h["cpu"]) for h in history], color="tab:red", lw=2.5, label="średnie CPU podów")
    ax2.axhline(HPA_TARGET, color="gray", ls="--", label=f"cel HPA = {HPA_TARGET}%")
    ax2.set_ylabel("CPU [% requests]")
    ax2.set_xlabel("czas [s]")
    ax2.set_ylim(bottom=0)

    prev = None
    for x, h in zip(xs, history):
        if h["note"] != prev:
            for ax in (ax1, ax2):
                ax.axvline(x, color="k", alpha=0.3, ls=":")
            ax1.annotate(h["note"], (x, 1), xycoords=("data", "axes fraction"),
                         xytext=(4, -14), textcoords="offset points", fontsize=9, fontweight="bold")
            prev = h["note"]
    for ax in (ax1, ax2):
        ax.grid(alpha=0.3)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.suptitle("HorizontalPodAutoscaler: obciążenie vs liczba replik")
    plt.tight_layout()
    plt.show()

def set_load(rps, n):
    """Ustawia generator ruchu: rps żądań/s, każde liczy fib(n). rps=0 wyłącza ruch."""
    if rps <= 0:
        kubectl("scale", "deployment/load-generator", "--replicas=0")
        return
    kubectl("set", "env", "deployment/load-generator", f"RPS={rps}", f"FIB_N={n}")
    kubectl("scale", "deployment/load-generator", "--replicas=1")

def load_info():
    d = kjson("get", "deployment", "load-generator")
    if not d["spec"]["replicas"]:
        return "wyłączony"
    env = {e["name"]: e["value"] for e in d["spec"]["template"]["spec"]["containers"][0].get("env", [])}
    return f"{env.get('RPS')} req/s × fib({env.get('FIB_N')})"

def monitor_hpa(seconds, note, interval=5):
    t_end = time.time() + seconds
    while True:
        st = kjson("get", "hpa", "fib-app").get("status", {})
        cpu = None
        for m in st.get("currentMetrics") or []:
            if m.get("type") == "Resource":
                cpu = m["resource"]["current"].get("averageUtilization")
        out, pods = render_pods("app=fib-app")
        ready = sum(pod_state(p) == "Ready" for p in pods)
        history.append(dict(t=time.time(), cpu=cpu, current=st.get("currentReplicas"),
                            desired=st.get("desiredReplicas"), ready=ready, note=note))
        clear_output(wait=True)
        plot_history()
        print(f"[{note}]  ruch: {load_info()}   CPU: {cpu}%   repliki: {st.get('currentReplicas')} → {st.get('desiredReplicas')}"
              f"   Ready: {ready}   (pozostało {max(0, t_end - time.time()):.0f}s)\n")
        print(out)
        if time.time() >= t_end:
            return
        time.sleep(interval)

kubectl("apply", "-f", "k8s-demo/hpa.yaml")
print(kubectl("get", "hpa"))

### Faza 1: brak ruchu
Aplikacja nic nie robi, więc HPA zejdzie do `minReplicas: 1`.
(Pierwsze odczyty CPU mogą mieć wartość `None`/`<unknown>`, zanim metrics-server zbierze dane.)

In [ ]:
monitor_hpa(75, "bez ruchu")

### Faza 2: umiarkowany ruch, 5 req/s × fib(25)

In [ ]:
set_load(rps=5, n=25)
monitor_hpa(150, "5/s × fib(25)")

### Faza 3: cięższe zapytania, ta sama liczba żądań, ale fib(27)
Liczba żądań się nie zmienia, a każde kosztuje ok. **2,6×** więcej CPU (1,6²). HPA dokłada pody.

In [ ]:
set_load(rps=5, n=27)
monitor_hpa(120, "5/s × fib(27)")

### Faza 4: nagły skok ruchu, 20 req/s × fib(27)
HPA dojdzie do `maxReplicas: 10`. Jeśli na węzłach zabraknie miejsca, pojawią się pody `Pending` ⚪
i do akcji wkroczy też Cluster Autoscaler (więcej w sekcji 6).

In [ ]:
set_load(rps=20, n=27)
monitor_hpa(150, "SKOK 20/s × fib(27)")

### Faza 5: koniec ruchu
Po 30 s okna stabilizacji HPA zacznie zwalniać pody (w produkcji okno jest dłuższe, żeby nie „pływać” przy wahaniach ruchu).

In [ ]:
set_load(rps=0, n=0)
monitor_hpa(120, "cisza")

> 🎛️ **Do eksperymentów na żywo:** `set_load(rps=..., n=...)` + `monitor_hpa(sekundy, "opis")`. Wykres dopisuje się do poprzednich faz.
> Liczby powyżej to punkt startowy. Szybkość CPU w różnych typach maszyn się różni, więc jeśli skalowanie jest za słabe
> lub za mocne, zmień `n` o ±1.

Decyzje HPA są zapisane w zdarzeniach (`Events`):

In [ ]:
%%bash
kubectl -n k8s-demo describe hpa fib-app | sed -n '/Events:/,$p'

> 💡 Kiedy działa HPA, ręczne `kubectl scale` zostanie wkrótce nadpisane, bo liczbą replik zarządza autoskaler.

---
## 6. Autoskalowanie węzłów: Cluster Autoscaler

HPA dodaje **pody**, ale nie **maszyny**. Scheduler umieści pod tylko tam, gdzie zostało wolne `requests` CPU/RAM.
Jeśli nigdzie nie ma miejsca, pod zostaje w stanie `Pending`. Wtedy **Cluster Autoscaler** w GKE tworzy nowe
maszyny wirtualne w puli węzłów (do `--max-nodes`), a gdy nie są już potrzebne, usuwa je.

Tworzymy pody, z których każdy rezerwuje 0,5 CPU, i jest ich więcej, niż zmieszczą obecne węzły:

In [ ]:
def cpu_cores(q):
    return int(q[:-1]) / 1000 if q.endswith("m") else float(q)

nodes_before = len(worker_nodes())
alloc = sum(cpu_cores(n["status"]["allocatable"]["cpu"]) for n in worker_nodes())
greedy = int(alloc / 0.5) + 4
print(f"{nodes_before} węzły, łącznie {alloc:.1f} CPU do zaalokowania, tworzymy {greedy} podów po 0,5 CPU")

kubectl("create", "deployment", "greedy", "--image=registry.k8s.io/pause:3.10", "--replicas=0")
kubectl("set", "resources", "deployment/greedy", "--requests=cpu=500m")
kubectl("scale", "deployment/greedy", f"--replicas={greedy}")
watch_pods("app=greedy", seconds=30, interval=2, title=f"🧱 {greedy} podów × 0,5 CPU")

In [ ]:
%%bash
echo "Dlaczego Pending? I co zrobił Cluster Autoscaler?"
kubectl -n k8s-demo get events --sort-by=.lastTimestamp \
  --field-selector involvedObject.kind=Pod -o custom-columns=REASON:.reason,MESSAGE:.message \
  | grep -E "FailedScheduling|TriggeredScaleUp|NotTriggerScaleUp" | tail -4

Obserwujemy, jak pojawiają się nowe węzły (zwykle 1,5–3 minuty: GKE tworzy VM, startuje kubelet, węzeł dołącza do klastra).
Pody `Pending` ⚪ trafiają na nowe węzły.

In [ ]:
watch_pods("app=greedy", seconds=300, interval=3, target=greedy, title="☁️ Cluster Autoscaler dokłada węzły")
print(f"\nWęzły: {nodes_before} → {len(worker_nodes())}")

In [ ]:
%%bash
gcloud compute instances list --project "$PROJECT" --filter="name~^gke-$CLUSTER" \
  --format="table(name, zone.basename(), machineType.basename(), status, creationTimestamp.date('%H:%M:%S'))"

Usuwamy obciążenie. Węzły **nie znikną od razu**: Cluster Autoscaler czeka, aż węzeł będzie niewykorzystany przez ok. 10 minut,
i usuwa tylko te, które nie spadną poniżej `--min-nodes`. Można do tego zajrzeć na końcu pokazu (`kubectl get nodes`).

Trzy poziomy skalowania w Kubernetesie:

| Co skalujemy | Narzędzie | Kiedy |
|---|---|---|
| liczba podów | **HPA** (+ KEDA dla kolejek/zdarzeń) | rośnie ruch |
| zasoby jednego poda | **VPA** | źle dobrane `requests` |
| liczba węzłów | **Cluster Autoscaler** (GKE: też Node Auto-Provisioning / Autopilot) | pody `Pending` |

In [ ]:
kubectl("delete", "deployment", "greedy")
print("Usunięto deployment greedy")

---
## 7. Rolling update i rollback

Deployment `web` (nginx) ma 6 replik. Strategia `RollingUpdate` z `maxSurge: 2` (maks. 2 pody ponad stan) i
`maxUnavailable: 1` (maks. 1 pod niedostępny) wymienia pody **partiami**. Usługa działa przez cały czas.

In [ ]:
%%writefile k8s-demo/web.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  annotations:
    kubernetes.io/change-cause: "nginx 1.27"
spec:
  replicas: 6
  minReadySeconds: 3          # spowalnia rollout, żeby było go widać
  selector:
    matchLabels: {app: web}
  strategy:
    type: RollingUpdate
    rollingUpdate: {maxSurge: 2, maxUnavailable: 1}
  template:
    metadata:
      labels: {app: web}
    spec:
      containers:
        - name: nginx
          image: nginx:1.27-alpine
          ports: [{containerPort: 80}]
          readinessProbe:
            httpGet: {path: /, port: 80}
            periodSeconds: 2
          resources:
            requests: {cpu: 50m, memory: 32Mi}

In [ ]:
kubectl("apply", "-f", "k8s-demo/web.yaml")
watch_pods("app=web", seconds=120, target=6, show_image=True, title="🚀 web v1.27")

### Aktualizacja do nowej wersji

In [ ]:
kubectl("set", "image", "deployment/web", "nginx=nginx:1.28-alpine")
kubectl("annotate", "deployment/web", "kubernetes.io/change-cause=nginx 1.28", "--overwrite")
watch_pods("app=web", seconds=120, target=6, show_image=True, title="🔄 Rolling update 1.27 → 1.28")

### Wdrażamy zepsutą wersję
Obraz nie istnieje. Nowe pody się nie uruchomią, ale **Kubernetes nie wyłączy starych**, bo `maxUnavailable: 1`
gwarantuje, że większość replik dalej obsługuje ruch.

In [ ]:
kubectl("set", "image", "deployment/web", "nginx=nginx:9.99-nie-istnieje")
kubectl("annotate", "deployment/web", "kubernetes.io/change-cause=nginx 9.99 (zepsuta)", "--overwrite")
watch_pods("app=web", seconds=30, show_image=True, title="💣 Wdrożenie zepsutej wersji")

In [ ]:
%%bash
kubectl -n k8s-demo get deployment web
echo
kubectl -n k8s-demo rollout history deployment/web

### Rollback jedną komendą

In [ ]:
kubectl("rollout", "undo", "deployment/web")
watch_pods("app=web", seconds=90, target=6, show_image=True, title="⏪ kubectl rollout undo")
print()
print(kubectl("rollout", "history", "deployment/web"))

---
## Podsumowanie

| Funkcja | Mechanizm | Co widzieliśmy |
|---|---|---|
| Samonaprawa | ReplicaSet + pętla kontrolna | usunięty pod / wyłączony węzeł → pody odtworzone w kilka sekund |
| Load balancing | Service `LoadBalancer` | publiczny IP od Google Cloud, ruch rozkłada się na wszystkie repliki |
| Skalowanie ręczne | `kubectl scale` | 3 → 12 replik jedną komendą |
| Autoskalowanie | HPA + metrics-server | więcej lub cięższych żądań fib(n) → więcej replik, cisza → mniej |
| Skalowanie węzłów | Cluster Autoscaler (GKE) | pody `Pending` → GKE sam dokłada maszyny wirtualne |
| Wdrożenia | RollingUpdate | nowa wersja bez przestoju, zepsuta nie kładzie usługi, rollback jedną komendą |

---
## 8. Sprzątanie

Najpierw usuwamy namespace. Usunięcie Service typu LoadBalancer zwalnia load balancer i publiczny IP w GCP.

In [ ]:
%%bash
kubectl delete namespace k8s-demo --timeout=300s

**Usuwamy klaster** (ok. 3–5 minut). Bez tego węzły dalej generują koszty!

In [ ]:
%%bash
gcloud container clusters delete "$CLUSTER" --zone "$ZONE" --project "$PROJECT" --quiet
echo
echo "Pozostałe VM klastra (powinno być pusto):"
gcloud compute instances list --project "$PROJECT" --filter="name~^gke-$CLUSTER" --format="value(name)"
echo "Pozostałe forwarding rules (load balancery):"
gcloud compute forwarding-rules list --project "$PROJECT" --format="value(name,IPAddress)"